This notebook aims to filter poor quality data to prepare data for actual evapotranspiration estimation from eddy covariance data and also filter 

In [31]:
# import packages
import pandas as pd
import numpy as np
import os

## Functions

### EBC functions

In [35]:
# implement energy balance correction for hlaf hourly data and see the results
from scipy.stats import iqr
import copy

def EBC_half_hourly_data(half_hourly_filtered_df,
                         flux_cols_dict={'Rn':'NETRAD_1_1_1', 'G':'G_1_1_1','LE':'LE', 'H':'H'}):

   # Ensure index is in datetime format
    df = half_hourly_filtered_df
    df = df.sort_index()
    df['hour'] = df.index.hour
    df['minute'] = df.index.minute
    df['time_of_day'] = df['hour'] + df['minute'] / 60  # Convert to decimal hour

    # Step 1: Compute EBC_CF for all half-hourly values
    df['EBC_CF'] = (df[flux_cols_dict['Rn']] - df[flux_cols_dict['G']]) / (df[flux_cols_dict['H']] + df[flux_cols_dict['LE']])

    # Step 2: Remove Outliers (Beyond 1.5× IQR)
    Q1, Q3 = df['EBC_CF'].quantile([0.25, 0.75])
    IQR_value = Q3 - Q1
    df['EBC_CF'] = np.where(
        (df['EBC_CF'] < (Q1 - 1.5 * IQR_value)) | (df['EBC_CF'] > (Q3 + 1.5 * IQR_value)),
        np.nan,
        df['EBC_CF']
    )
    
    # Step 3: Compute Smoothed EBC_CF Over ±15 Days for Each Half-Hour Time Step (Method 1)
    def compute_EBC_CF_Method1(timestamp):
        
        """Computes median of values within ±15-day window, filtering between 10:00-14:30 and 22:00-02:30."""
        df_window = df.loc[timestamp - pd.Timedelta(days=15) : timestamp + pd.Timedelta(days=15)]

        # Select values in the two time ranges
        valid_values = pd.concat([
            df_window.between_time("10:00", "14:30"),
            df_window.between_time("22:00", "02:30")
        ])["EBC_CF"].dropna()

        return valid_values.median() if len(valid_values) >= 5 else np.nan
    
    df["EBC_CF_Method1"] = df.index.to_series().apply(compute_EBC_CF_Method1)

    # Step 4: Use Method 2 when "EBC_CF_Method1" is np.nan
    def compute_EBC_CF_Method2(timestamp):
        """ Function to compute Method 2 where Method 1 fails"""
        return (df.loc[timestamp - pd.Timedelta(days=5) : timestamp + pd.Timedelta(days=5)]
                  .between_time("10:00", "14:30")
                  .between_time((timestamp - pd.Timedelta(hours=1)).strftime("%H:%M"),
                                (timestamp + pd.Timedelta(hours=1)).strftime("%H:%M"))["EBC_CF"]
                  .dropna()
                  .mean() if len(df) >= 5 else np.nan)

    df["EBC_CF_Method2"] = df[df["EBC_CF_Method1"].isna()].index.to_series().apply(compute_EBC_CF_Method2)
    
    # Step 5: Final EBC_CF selection: Use Method 1 where available, otherwise use Method 2
    df["EBC_CF_Final"] = df["EBC_CF_Method1"].combine_first(df["EBC_CF_Method2"])
    
    # Step 6: Apply the Corrected EBC_CF to All Data Points
    df['H_corr'] = df[flux_cols_dict['H']] * df['EBC_CF_Final']
    df['LE_corr'] = df[flux_cols_dict['LE']] * df['EBC_CF_Final']

    # Compute Corrected Energy Imbalance (Should be Close to Zero)
    df['Imb_corr'] = df[flux_cols_dict['Rn']] - df[flux_cols_dict['G']] - (df['H_corr'] + df['LE_corr'])

    return df

In [36]:
def EBC_half_hourly_data(half_hourly_filtered_df,
                         flux_cols_dict={'Rn':'NETRAD_1_1_1', 'G':'G_1_1_1','LE':'LE', 'H':'H'}):

   # Ensure index is in datetime format
    df = half_hourly_filtered_df
    df = df.sort_index()
    df['hour'] = df.index.hour
    df['minute'] = df.index.minute
    df['time_of_day'] = df['hour'] + df['minute'] / 60  # Convert to decimal hour

    # Step 1: Compute EBC_CF for all half-hourly values
    df['EBC_CF'] = (df[flux_cols_dict['Rn']] - df[flux_cols_dict['G']]) / (df[flux_cols_dict['H']] + df[flux_cols_dict['LE']])

    # Step 2: Remove Outliers (Beyond 1.5× IQR)
    Q1, Q3 = df['EBC_CF'].quantile([0.25, 0.75])
    IQR_value = Q3 - Q1
    df['EBC_CF'] = np.where(
        (df['EBC_CF'] < (Q1 - 1.5 * IQR_value)) | (df['EBC_CF'] > (Q3 + 1.5 * IQR_value)),
        np.nan,
        df['EBC_CF']
    )
    
    # Step 3: Compute Smoothed EBC_CF Over ±15 Days for Each Half-Hour Time Step (Method 1)
    def compute_EBC_CF_Method1(timestamp):
        
        """Computes median of values within ±15-day window, filtering between 10:00-14:30 and 22:00-02:30."""
        df_window = df.loc[timestamp - pd.Timedelta(days=15) : timestamp + pd.Timedelta(days=15)]

        # Select values in the two time ranges
        valid_values = pd.concat([
            df_window.between_time("10:00", "14:30"),
            df_window.between_time("22:00", "02:30")
        ])["EBC_CF"].dropna()

        return valid_values.median() if len(valid_values) >= 5 else np.nan
    
    df["EBC_CF_Method1"] = df.index.to_series().apply(compute_EBC_CF_Method1)

    # Step 4: Use Method 2 when "EBC_CF_Method1" is np.nan
    def compute_EBC_CF_Method2(timestamp):
        """ Function to compute Method 2 where Method 1 fails"""
        return (df.loc[timestamp - pd.Timedelta(days=5) : timestamp + pd.Timedelta(days=5)]
                  .between_time("10:00", "14:30")
                  .between_time((timestamp - pd.Timedelta(hours=1)).strftime("%H:%M"),
                                (timestamp + pd.Timedelta(hours=1)).strftime("%H:%M"))["EBC_CF"]
                  .dropna()
                  .mean() if len(df) >= 5 else np.nan)

    df["EBC_CF_Method2"] = df[df["EBC_CF_Method1"].isna()].index.to_series().apply(compute_EBC_CF_Method2)
    
    # Step 5: Final EBC_CF selection: Use Method 1 where available, otherwise use Method 2
    df["EBC_CF_Final"] = df["EBC_CF_Method1"].combine_first(df["EBC_CF_Method2"])
    
    # Step 6: Apply the Corrected EBC_CF to All Data Points
    df['H_corr'] = df[flux_cols_dict['H']] * df['EBC_CF_Final']
    df['LE_corr'] = df[flux_cols_dict['LE']] * df['EBC_CF_Final']

    # Compute Corrected Energy Imbalance (Should be Close to Zero)
    df['Imb_corr'] = df[flux_cols_dict['Rn']] - df[flux_cols_dict['G']] - (df['H_corr'] + df['LE_corr'])

    return df

In [38]:
def gapfill_LE_H(df,
                flux_cols_dict={'Rn':'NETRAD_1_1_1', 'G':'G_1_1_1','LE':'LE', 'H':'H'}):
    """
    Gap-fills LE and H using linear interpolation:
    - Daytime (Rn > 0): Max 2-hour gaps (4 half-hour periods).
    - Nighttime (Rn ≤ 0): Max 4-hour gaps (8 half-hour periods).

    Parameters:
    df (pd.DataFrame): DataFrame with 'LE', 'H', and 'Rn'.

    Returns:
    pd.DataFrame: DataFrame with 'LE_filled' and 'H_filled'.
    """
    df = df.copy()
    day = df[flux_cols_dict["Rn"]] > 0  # Daytime mask
    
    for col in [flux_cols_dict["LE"], flux_cols_dict["H"]]:
        #df[f"{col}_F"] = df[col]
        df[f"{col}_F"] = df[col].interpolate(limit=2).where(day, df[col].interpolate(limit=4))

    return df

## Code

#### Preprocess half hourly data

In [39]:
# read the half hourly data for all the seasons (3 towers with 9 seasons)
# PA
USUC1_path = os.path.join(os.getcwd(), 'EC_raw_data', 'AMF_US-UC1_BASE_HH_5-5.csv')
USUC2_path = os.path.join(os.getcwd(), 'EC_raw_data', 'AMF_US-UC2_BASE_HH_5-5.csv')
USHWB_path = os.path.join(os.getcwd(), 'EC_raw_data', 'AMF_US-HWB_BASE_HH_2-5.csv')
USUC1_df = pd.read_csv(USUC1_path,skiprows=[0,1] ,index_col = 'TIMESTAMP_START', parse_dates=['TIMESTAMP_START'])
USUC2_df = pd.read_csv(USUC2_path,skiprows=[0,1] ,index_col = 'TIMESTAMP_START', parse_dates=['TIMESTAMP_START'])
USHWB_df = pd.read_csv(USHWB_path,skiprows=[0,1] ,index_col = 'TIMESTAMP_START', parse_dates=['TIMESTAMP_START'])

# CA
USBi1_path = os.path.join(os.getcwd(), 'EC_raw_data', 'AMF_US-Bi1_BASE_HH_15-5.csv')
USBi2_path = os.path.join(os.getcwd(), 'EC_raw_data', 'AMF_US-Bi2_BASE_HH_20-5.csv')
USTw3_path = os.path.join(os.getcwd(), 'EC_raw_data', 'AMF_US-Tw3_BASE_HH_5-5.csv')
USBi1_df = pd.read_csv(USBi1_path,skiprows=[0,1] ,index_col = 'TIMESTAMP_START', parse_dates=['TIMESTAMP_START'])
USBi2_df = pd.read_csv(USBi2_path,skiprows=[0,1] ,index_col = 'TIMESTAMP_START', parse_dates=['TIMESTAMP_START'])
USTw3_df = pd.read_csv(USTw3_path,skiprows=[0,1] ,index_col = 'TIMESTAMP_START', parse_dates=['TIMESTAMP_START'])

# IL
USUiA_path = os.path.join(os.getcwd(), 'EC_raw_data', 'AMF_US-UiA_BASE_HH_3-5.csv')
USUiB_path = os.path.join(os.getcwd(), 'EC_raw_data', 'AMF_US-UiB_BASE_HH_3-5.csv')
USUiC_path = os.path.join(os.getcwd(), 'EC_raw_data', 'AMF_US-UiC_BASE_HH_2-5.csv')
USUiA_df = pd.read_csv(USUiA_path,skiprows=[0,1] ,index_col = 'TIMESTAMP_START', parse_dates=['TIMESTAMP_START'])
USUiB_df = pd.read_csv(USUiB_path,skiprows=[0,1] ,index_col = 'TIMESTAMP_START', parse_dates=['TIMESTAMP_START'])
USUiC_df = pd.read_csv(USUiC_path,skiprows=[0,1] ,index_col = 'TIMESTAMP_START', parse_dates=['TIMESTAMP_START'])

# IN (some column names need to be fixed)
USVT1_path = os.path.join(os.getcwd(), 'EC_raw_data', 'AMF_US-VT1_BASE_HH_2-5.csv')
USVT2_path = os.path.join(os.getcwd(), 'EC_raw_data', 'AMF_US-VT2_BASE_HH_2-5.csv')
USVT1_df = pd.read_csv(USVT1_path,skiprows=[0,1] ,index_col = 'TIMESTAMP_START', parse_dates=['TIMESTAMP_START'])
USVT2_df = pd.read_csv(USVT2_path,skiprows=[0,1] ,index_col = 'TIMESTAMP_START', parse_dates=['TIMESTAMP_START'])

In [4]:
## Check for the columns that are needed to rename them and remove the wrongly named col
df = USBi1_df
(
    df.replace(-9999, np.nan)  # treat -9999 as NaN
    .isna()
    .sum()
    .div(len(df))
    .mul(100)
    .round(2)
    .sort_values(ascending=True)
)

TIMESTAMP_END            0.00
TAU_SSITC_TEST_PI_F      6.60
FC_PI_F                  6.60
RECO_PI_F                6.60
LE_PI_F                  6.60
                        ...  
SWC_1_2_1               75.04
SWC_3_1_1               75.70
SWC_1_1_1               75.70
SWC_2_1_1               75.71
FCH4_PI_F              100.00
Length: 73, dtype: float64

In [40]:
# Convert  col names manually (This need to be done for each tower data df separately by manual inspection)
USUC1_df = USUC1_df.rename(columns={'NETRAD_1_1_1':'NETRAD', 'RH_1_1_1': 'RH', 'SW_IN_1_1_1': 'SW_IN', 'LW_IN_1_1_1': 'LW_IN', 'G_1_1_1': 'G',
                                     'P_RAIN_1_1_1': 'P_RAIN', 'TA_1_1_1': 'TA'})
USUC2_df = USUC2_df.rename(columns={'NETRAD_1_1_1':'NETRAD', 'RH_1_1_1': 'RH', 'SW_IN_1_1_1': 'SW_IN', 'LW_IN_1_1_1': 'LW_IN', 'G_1_1_1': 'G',
                                     'P_RAIN_1_1_1': 'P_RAIN', 'TA_1_1_1': 'TA'})
USHWB_df = USHWB_df.rename(columns={'NETRAD_1_1_1':'NETRAD', 'SW_IN_1_1_1': 'SW_IN', 'LW_IN_1_1_1': 'LW_IN', 'G_1_1_1': 'G',
                                     'P_RAIN_1_1_1': 'P_RAIN'})

USBi1_df = USBi1_df.rename(columns={'P': 'P_RAIN'})
USBi1_df = USBi1_df.rename(columns={'LE': 'LE_old', 'LE_PI_F':'LE'})
USBi1_df = USBi1_df.rename(columns={'H': 'H_old', 'H_PI_F':'H'})
USBi2_df = USBi2_df.rename(columns={'P': 'P_RAIN'})
USBi2_df = USBi2_df.rename(columns={'LE': 'LE_old', 'LE_PI_F':'LE'})
USBi2_df = USBi2_df.rename(columns={'H': 'H_old', 'H_PI_F':'H'})
USTw3_df = USTw3_df.rename(columns={'P': 'P_RAIN'})
USTw3_df = USTw3_df.rename(columns={'LE': 'LE_old', 'LE_PI_F':'LE'})
USTw3_df = USTw3_df.rename(columns={'H': 'H_old', 'H_PI_F':'H'})

USUiA_df = USUiA_df.rename(columns={'P': 'P_RAIN'})
USUiA_df = USUiA_df.rename(columns={'TA': 'TA_old', 'TA_1_1_1':'TA'})
USUiA_df = USUiA_df.rename(columns={'G': 'G_old', 'G_1_1_1':'G'})
USUiB_df = USUiB_df.rename(columns={'P': 'P_RAIN'})
USUiB_df = USUiB_df.rename(columns={'TA': 'TA_old', 'TA_1_1_1':'TA'})
USUiB_df = USUiB_df.rename(columns={'G': 'G_old', 'G_1_1_1':'G'})
USUiC_df = USUiC_df.rename(columns={'P': 'P_RAIN'})

USVT1_df = USVT1_df.rename(columns={'LE_1_1_1':'LE', 'RH_1_1_1': 'RH', 'SW_IN_1_1_1': 'SW_IN', 'LW_IN_1_1_1': 'LW_IN', 'WS_1_1_1':'WS', 'H_1_1_1':'H',
                                    'G_1_1_1': 'G', 'PA_1_1_1':'PA', 'P_1_1_1': 'P_RAIN', 'TA_1_1_1': 'TA',
                                    'WS_1_1_1': 'WS', 'WD_1_1_1': 'WD', 'V_SIGMA_1_1_1': 'V_SIGMA', 'USTAR_1_1_1': 'USTAR', 'MO_LENGTH_1_1_1': 'MO_LENGTH'})
USVT2_df = USVT2_df.rename(columns={'LE_1_1_1':'LE', 'RH_1_1_1': 'RH', 'SW_IN_1_1_1': 'SW_IN', 'LW_IN_1_1_1': 'LW_IN', 'WS_1_1_1':'WS', 'H_1_1_1':'H',
                                    'G_1_1_1': 'G', 'PA_1_1_1':'PA', 'P_1_1_1': 'P_RAIN', 'TA_1_1_1': 'TA',
                                    'WS_1_1_1': 'WS', 'WD_1_1_1': 'WD', 'V_SIGMA_1_1_1': 'V_SIGMA', 'USTAR_1_1_1': 'USTAR', 'MO_LENGTH_1_1_1': 'MO_LENGTH'})

In [41]:
# Only keep important cols needed for our analysis
cols_to_keep = ['NETRAD', 'LE', 'H', 'G', 'LE_SSITC_TEST', 'H_SSITC_TEST', 'TA', 'RH', 'SW_IN','LW_IN', 'PA','P_RAIN','WS', 'WD','V_SIGMA','USTAR', 'MO_LENGTH']

USUC1_df = USUC1_df[cols_to_keep]
USUC2_df = USUC2_df[cols_to_keep]
USHWB_df = USHWB_df[cols_to_keep]

USBi1_df = USBi1_df[cols_to_keep]
USBi2_df = USBi2_df[cols_to_keep]
USTw3_df = USTw3_df[cols_to_keep]

USUiA_df = USUiA_df[cols_to_keep]
USUiB_df = USUiB_df[cols_to_keep]
USUiC_df = USUiC_df[cols_to_keep]

USVT1_df = USVT1_df[cols_to_keep]
USVT2_df = USVT2_df[cols_to_keep]


In [43]:
USTw3_df.loc['2017':]

,NETRAD,LE,H,G,LE_SSITC_TEST,H_SSITC_TEST,TA,RH,SW_IN,LW_IN,PA,P_RAIN,WS,WD,V_SIGMA,USTAR,MO_LENGTH
TIMESTAMP_START,,,,,,,,,,,,,,,,,
2017-01-01 00:00:00,-9999.000000,3.721116,-6.823849,-26.823114,2,1,3.845,89.20,-3.150788,-9999.000000,101.556,-9999.0,2.201256,251.840538,0.256233,0.097220,8.432498
2017-01-01 00:30:00,-9999.000000,8.444940,-16.611317,-26.196009,2,1,3.734,87.40,-3.600900,-9999.000000,101.537,-9999.0,2.554735,255.356768,0.261480,0.132919,13.201932
2017-01-01 01:00:00,-9999.000000,9.922046,-17.040919,-27.448277,2,1,3.299,88.40,-2.550638,-9999.000000,101.546,-9999.0,2.252223,258.095470,0.295347,0.151389,19.125140
2017-01-01 01:30:00,-9999.000000,4.605690,-1.220315,-21.396803,2,2,4.054,88.80,0.525131,-9999.000000,101.562,-9999.0,0.395825,18.201816,0.825390,0.085506,20.149969
2017-01-01 02:00:00,-9999.000000,5.978117,-16.683441,-13.956871,2,2,5.032,88.40,0.300075,-9999.000000,101.590,-9999.0,0.825168,56.321194,0.447772,0.116346,8.603983
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2018-06-04 21:30:00,-101.419775,10.413375,-71.022176,-12.791707,2,0,16.470,53.21,-4.201050,288.270052,101.073,0.0,4.964423,243.374469,0.772418,0.392501,78.166396
2018-06-04 22:00:00,-102.341304,10.445250,-70.693179,-13.123959,2,1,16.180,54.53,-4.201050,286.811845,101.057,0.0,5.366806,245.677892,0.855991,0.472747,137.243865
2018-06-04 22:30:00,-101.860831,9.304259,-62.463735,-13.460477,1,1,15.880,56.72,-4.276069,285.484987,101.057,0.0,5.090994,243.339991,0.775806,0.383163,82.736907


In [44]:
# for each half hourly data check the flags and drop those rows with LE flag of 2
Rn_col = 'NETRAD'
LE_col = 'LE'
H_col = 'H'
LE_QC_flag_col = 'LE_SSITC_TEST'
H_QC_flag_col = 'H_SSITC_TEST'

# keep only LE values when both LE and H flags are not 2
# PA
USUC1_df[LE_col] = USUC1_df.apply(lambda row: row[LE_col] if (row[LE_QC_flag_col]==0) & (row[H_QC_flag_col]==0) else np.nan, axis=1)
USUC2_df[LE_col] = USUC2_df.apply(lambda row: row[LE_col] if (row[LE_QC_flag_col]==0) & (row[H_QC_flag_col]==0) else np.nan, axis=1)
USHWB_df[LE_col] = USHWB_df.apply(lambda row: row[LE_col] if (row[LE_QC_flag_col]==0) & (row[H_QC_flag_col]==0) else np.nan, axis=1)

# CA (These are PI corrected values so no need for Foken flag application)
USBi1_df[LE_col] = USBi1_df.apply(lambda row: row[LE_col] if (row[LE_QC_flag_col]!=3) & (row[H_QC_flag_col]!=3) else np.nan, axis=1)
USBi2_df[LE_col] = USBi2_df.apply(lambda row: row[LE_col] if (row[LE_QC_flag_col]!=3) & (row[H_QC_flag_col]!=3) else np.nan, axis=1)
USTw3_df[LE_col] = USTw3_df.apply(lambda row: row[LE_col] if (row[LE_QC_flag_col]!=3) & (row[H_QC_flag_col]!=3) else np.nan, axis=1)

# IL
USUiA_df[LE_col] = USUiA_df.apply(lambda row: row[LE_col] if (row[LE_QC_flag_col]!=2) & (row[H_QC_flag_col]!=2) else np.nan, axis=1)
USUiB_df[LE_col] = USUiB_df.apply(lambda row: row[LE_col] if (row[LE_QC_flag_col]!=2) & (row[H_QC_flag_col]!=2) else np.nan, axis=1)
USUiC_df[LE_col] = USUiC_df.apply(lambda row: row[LE_col] if (row[LE_QC_flag_col]!=2) & (row[H_QC_flag_col]!=2) else np.nan, axis=1)

#IN
USVT1_df[LE_col] = USVT1_df.apply(lambda row: row[LE_col] if (row[LE_QC_flag_col]!=2) & (row[H_QC_flag_col]!=2) else np.nan, axis=1)
USVT2_df[LE_col] = USVT2_df.apply(lambda row: row[LE_col] if (row[LE_QC_flag_col]!=2) & (row[H_QC_flag_col]!=2) else np.nan, axis=1)


# keep only H values when both LE and H flags are not 2
# PA
USUC1_df[H_col] = USUC1_df.apply(lambda row: row[H_col] if (row[LE_QC_flag_col]!=2) & (row[H_QC_flag_col]!=2) else np.nan, axis=1)
USUC2_df[H_col] = USUC2_df.apply(lambda row: row[H_col] if (row[LE_QC_flag_col]!=2) & (row[H_QC_flag_col]!=2) else np.nan, axis=1)
USHWB_df[H_col] = USHWB_df.apply(lambda row: row[H_col] if (row[LE_QC_flag_col]!=2) & (row[H_QC_flag_col]!=2) else np.nan, axis=1)

# CA (These are PI corrected values so no need for Foken flag application)
USBi1_df[H_col] = USBi1_df.apply(lambda row: row[H_col] if (row[LE_QC_flag_col]!=3) & (row[H_QC_flag_col]!=3) else np.nan, axis=1)
USBi2_df[H_col] = USBi2_df.apply(lambda row: row[H_col] if (row[LE_QC_flag_col]!=3) & (row[H_QC_flag_col]!=3) else np.nan, axis=1)
USTw3_df[H_col] = USTw3_df.apply(lambda row: row[H_col] if (row[LE_QC_flag_col]!=3) & (row[H_QC_flag_col]!=3) else np.nan, axis=1)

# IL
USUiA_df[H_col] = USUiA_df.apply(lambda row: row[H_col] if (row[LE_QC_flag_col]!=2) & (row[H_QC_flag_col]!=2) else np.nan, axis=1)
USUiB_df[H_col] = USUiB_df.apply(lambda row: row[H_col] if (row[LE_QC_flag_col]!=2) & (row[H_QC_flag_col]!=2) else np.nan, axis=1)
USUiC_df[H_col] = USUiC_df.apply(lambda row: row[H_col] if (row[LE_QC_flag_col]!=2) & (row[H_QC_flag_col]!=2) else np.nan, axis=1)

# IN
USVT1_df[H_col] = USVT1_df.apply(lambda row: row[H_col] if (row[LE_QC_flag_col]!=2) & (row[H_QC_flag_col]!=2) else np.nan, axis=1)
USVT2_df[H_col] = USVT2_df.apply(lambda row: row[H_col] if (row[LE_QC_flag_col]!=2) & (row[H_QC_flag_col]!=2) else np.nan, axis=1)


# filter negative LE during daylight
# PA
USUC1_df[LE_col] = USUC1_df.apply(lambda row: np.nan if (row[Rn_col]>0) & (row[LE_col]<0) else row[LE_col], axis=1)
USUC2_df[LE_col] = USUC2_df.apply(lambda row: np.nan if (row[Rn_col]>0) & (row[LE_col]<0) else row[LE_col], axis=1)
USHWB_df[LE_col] = USHWB_df.apply(lambda row: np.nan if (row[Rn_col]>0) & (row[LE_col]<0) else row[LE_col], axis=1)

# CA

USBi1_df[LE_col] = USBi1_df.apply(lambda row: np.nan if (row[Rn_col]>0) & (row[LE_col]<0) else row[LE_col], axis=1)
USBi2_df[LE_col] = USBi2_df.apply(lambda row: np.nan if (row[Rn_col]>0) & (row[LE_col]<0) else row[LE_col], axis=1)
USTw3_df[LE_col] = USTw3_df.apply(lambda row: np.nan if (row[Rn_col]>0) & (row[LE_col]<0) else row[LE_col], axis=1)

# IL
USUiA_df[LE_col] = USUiA_df.apply(lambda row: np.nan if (row[Rn_col]>0) & (row[LE_col]<0) else row[LE_col], axis=1)
USUiB_df[LE_col] = USUiB_df.apply(lambda row: np.nan if (row[Rn_col]>0) & (row[LE_col]<0) else row[LE_col], axis=1)
USUiC_df[LE_col] = USUiC_df.apply(lambda row: np.nan if (row[Rn_col]>0) & (row[LE_col]<0) else row[LE_col], axis=1)

# IN
USVT1_df[LE_col] = USVT1_df.apply(lambda row: np.nan if (row[Rn_col]>0) & (row[LE_col]<0) else row[LE_col], axis=1)
USVT2_df[LE_col] = USVT2_df.apply(lambda row: np.nan if (row[Rn_col]>0) & (row[LE_col]<0) else row[LE_col], axis=1)


# now divide the data into growing seasons and only keep the growing season period with 15 days of buffer for energy balance correction steps
# Define the time windows for growing seasons
# PA
US_UC_growing_season_2019 = [('2019-05-20', '2019-09-11')]
US_UC_growing_season_2020 = [('2020-05-27', '2020-09-25')]
US_UC_growing_season_2021 = [('2021-05-20', '2021-11-16')]
US_UC_growing_season_2022 = [('2022-06-08', '2022-12-08')]
US_UC_growing_season_2023 = [('2023-06-02', '2023-10-26')]
US_UC_growing_season_2024 = [('2024-05-25', '2024-10-08')]
US_HWB_growing_season_2016 = [('2016-03-15', '2016-10-12')]
US_HWB_growing_season_2017 = [('2017-04-15', '2017-09-30')]

# CA
US_Bi1_growing_season_2018 = [('2018-02-05', '2018-10-25')]
US_Bi1_growing_season_2019 = [('2019-02-05', '2019-10-25')]
US_Bi1_growing_season_2020 = [('2020-02-05', '2020-10-25')]
US_Bi1_growing_season_2021 = [('2021-02-05', '2021-10-25')]
US_Bi1_growing_season_2022 = [('2022-02-05', '2022-10-25')]
US_Bi1_growing_season_2023 = [('2023-02-05', '2023-10-25')]
US_Bi1_growing_season_2024 = [('2024-02-05', '2024-09-25')]
US_Bi2_growing_season_2018 = [('2018-05-26', '2018-09-22')]
US_Bi2_growing_season_2019 = [('2019-05-25', '2019-09-14')]
US_Bi2_growing_season_2020 = [('2020-04-30', '2020-10-12')]
US_Bi2_growing_season_2021 = [('2021-05-03', '2021-10-13')]
US_Bi2_growing_season_2022 = [('2022-05-07', '2022-10-13')]
US_Bi2_growing_season_2023 = [('2023-06-01', '2023-10-27')]
US_Bi2_growing_season_2024 = [('2024-05-09', '2024-10-10')]
US_Tw3_growing_season_2017 = [('2017-02-05', '2017-09-26')]
US_Tw3_growing_season_2018 = [('2018-02-05', '2018-06-01')]

# IL
US_UiA_growing_season_2017 = [('2017-05-15', '2017-09-22')]
US_UiA_growing_season_2018 = [('2018-05-26', '2018-09-22')]
US_UiA_growing_season_2019 = [('2019-05-25', '2019-09-14')]
US_UiA_growing_season_2020 = [('2020-04-30', '2020-10-12')]
US_UiA_growing_season_2021 = [('2021-05-03', '2021-10-13')]
US_UiA_growing_season_2022 = [('2022-05-07', '2022-10-13')]
US_UiA_growing_season_2023 = [('2023-06-01', '2023-10-27')]
US_UiA_growing_season_2024 = [('2024-05-09', '2024-10-10')]
US_UiB_growing_season_2017 = [('2017-05-15', '2017-09-22')]
US_UiB_growing_season_2018 = [('2018-05-26', '2018-09-22')]
US_UiB_growing_season_2019 = [('2019-05-25', '2019-09-14')]
US_UiB_growing_season_2020 = [('2020-04-30', '2020-10-12')]
US_UiB_growing_season_2021 = [('2021-05-03', '2021-10-13')]
US_UiB_growing_season_2022 = [('2022-05-07', '2022-10-13')]
US_UiB_growing_season_2023 = [('2023-06-01', '2023-10-27')]
US_UiB_growing_season_2024 = [('2024-05-09', '2024-10-10')]
US_UiC_growing_season_2017 = [('2017-05-15', '2017-09-22')]
US_UiC_growing_season_2018 = [('2018-05-26', '2018-09-22')]
US_UiC_growing_season_2019 = [('2019-05-25', '2019-09-14')]
US_UiC_growing_season_2020 = [('2020-04-30', '2020-10-12')]
US_UiC_growing_season_2021 = [('2021-05-03', '2021-10-13')]
US_UiC_growing_season_2022 = [('2022-05-07', '2022-10-13')]
US_UiC_growing_season_2023 = [('2023-06-01', '2023-10-27')]
US_UiC_growing_season_2024 = [('2024-05-09', '2024-10-10')]

# IN
US_VT1_growing_season_2023 = [('2023-05-05', '2023-10-25')]
US_VT1_growing_season_2024 = [('2024-05-05', '2024-09-25')]
US_VT2_growing_season_2023 = [('2023-05-05', '2023-10-25')]
US_VT2_growing_season_2024 = [('2024-05-05', '2024-09-25')]


def filter_time_periods(df, time_periods, buffer_days=15):
    buffer = pd.Timedelta(days=buffer_days)
    return df.loc[pd.Timestamp(time_periods[0][0]) - buffer : pd.Timestamp(time_periods[0][1]) + buffer]

# PA
USUC1_df_LE_filtered_2019 = filter_time_periods(USUC1_df, US_UC_growing_season_2019)
USUC1_df_LE_filtered_2020 = filter_time_periods(USUC1_df, US_UC_growing_season_2020)
USUC1_df_LE_filtered_2021 = filter_time_periods(USUC1_df, US_UC_growing_season_2021)
USUC1_df_LE_filtered_2022 = filter_time_periods(USUC1_df, US_UC_growing_season_2022)
USUC1_df_LE_filtered_2023 = filter_time_periods(USUC1_df, US_UC_growing_season_2023)
USUC1_df_LE_filtered_2024 = filter_time_periods(USUC1_df, US_UC_growing_season_2024)
USUC2_df_LE_filtered_2019 = filter_time_periods(USUC2_df, US_UC_growing_season_2019)
USUC2_df_LE_filtered_2020 = filter_time_periods(USUC2_df, US_UC_growing_season_2020)
USUC2_df_LE_filtered_2021 = filter_time_periods(USUC2_df, US_UC_growing_season_2021)
USUC2_df_LE_filtered_2022 = filter_time_periods(USUC2_df, US_UC_growing_season_2022)
USUC2_df_LE_filtered_2023 = filter_time_periods(USUC2_df, US_UC_growing_season_2023)
USUC2_df_LE_filtered_2024 = filter_time_periods(USUC2_df, US_UC_growing_season_2024)
USHWB_df_LE_filtered_2016 = filter_time_periods(USHWB_df, US_HWB_growing_season_2016)
USHWB_df_LE_filtered_2017 = filter_time_periods(USHWB_df, US_HWB_growing_season_2017)

# CA
USBi1_df_LE_filtered_2018 = filter_time_periods(USBi1_df, US_Bi1_growing_season_2018)
USBi1_df_LE_filtered_2019 = filter_time_periods(USBi1_df, US_Bi1_growing_season_2019)
USBi1_df_LE_filtered_2020 = filter_time_periods(USBi1_df, US_Bi1_growing_season_2020)
USBi1_df_LE_filtered_2021 = filter_time_periods(USBi1_df, US_Bi1_growing_season_2021)
USBi1_df_LE_filtered_2022 = filter_time_periods(USBi1_df, US_Bi1_growing_season_2022)
USBi1_df_LE_filtered_2023 = filter_time_periods(USBi1_df, US_Bi1_growing_season_2023)
USBi1_df_LE_filtered_2024 = filter_time_periods(USBi1_df, US_Bi1_growing_season_2024)
USBi2_df_LE_filtered_2018 = filter_time_periods(USBi2_df, US_Bi2_growing_season_2018)
USBi2_df_LE_filtered_2019 = filter_time_periods(USBi2_df, US_Bi2_growing_season_2019)
USBi2_df_LE_filtered_2020 = filter_time_periods(USBi2_df, US_Bi2_growing_season_2020)
USBi2_df_LE_filtered_2021 = filter_time_periods(USBi2_df, US_Bi2_growing_season_2021)
USBi2_df_LE_filtered_2022 = filter_time_periods(USBi2_df, US_Bi2_growing_season_2022)
USBi2_df_LE_filtered_2023 = filter_time_periods(USBi2_df, US_Bi2_growing_season_2023)
USBi2_df_LE_filtered_2024 = filter_time_periods(USBi2_df, US_Bi2_growing_season_2024)
USTw3_df_LE_filtered_2017 = filter_time_periods(USTw3_df, US_Tw3_growing_season_2017)
USTw3_df_LE_filtered_2018 = filter_time_periods(USTw3_df, US_Tw3_growing_season_2018)

# IL
USUiA_df_LE_filtered_2017 = filter_time_periods(USUiA_df, US_UiA_growing_season_2017)
USUiA_df_LE_filtered_2018 = filter_time_periods(USUiA_df, US_UiA_growing_season_2018)
USUiA_df_LE_filtered_2019 = filter_time_periods(USUiA_df, US_UiA_growing_season_2019)
USUiA_df_LE_filtered_2020 = filter_time_periods(USUiA_df, US_UiA_growing_season_2020)
USUiA_df_LE_filtered_2021 = filter_time_periods(USUiA_df, US_UiA_growing_season_2021)
USUiA_df_LE_filtered_2022 = filter_time_periods(USUiA_df, US_UiA_growing_season_2022)
USUiA_df_LE_filtered_2023 = filter_time_periods(USUiA_df, US_UiA_growing_season_2023)
USUiA_df_LE_filtered_2024 = filter_time_periods(USUiA_df, US_UiA_growing_season_2024)
USUiB_df_LE_filtered_2017 = filter_time_periods(USUiB_df, US_UiB_growing_season_2017)
USUiB_df_LE_filtered_2018 = filter_time_periods(USUiB_df, US_UiB_growing_season_2018)
USUiB_df_LE_filtered_2019 = filter_time_periods(USUiB_df, US_UiB_growing_season_2019)
USUiB_df_LE_filtered_2020 = filter_time_periods(USUiB_df, US_UiB_growing_season_2020)
USUiB_df_LE_filtered_2021 = filter_time_periods(USUiB_df, US_UiB_growing_season_2021)
USUiB_df_LE_filtered_2022 = filter_time_periods(USUiB_df, US_UiB_growing_season_2022)
USUiB_df_LE_filtered_2023 = filter_time_periods(USUiB_df, US_UiB_growing_season_2023)
USUiB_df_LE_filtered_2024 = filter_time_periods(USUiB_df, US_UiB_growing_season_2024)
USUiC_df_LE_filtered_2017 = filter_time_periods(USUiC_df, US_UiC_growing_season_2017)
USUiC_df_LE_filtered_2018 = filter_time_periods(USUiC_df, US_UiC_growing_season_2018)
USUiC_df_LE_filtered_2019 = filter_time_periods(USUiC_df, US_UiC_growing_season_2019)
USUiC_df_LE_filtered_2020 = filter_time_periods(USUiC_df, US_UiC_growing_season_2020)
USUiC_df_LE_filtered_2021 = filter_time_periods(USUiC_df, US_UiC_growing_season_2021)
USUiC_df_LE_filtered_2022 = filter_time_periods(USUiC_df, US_UiC_growing_season_2022)
USUiC_df_LE_filtered_2023 = filter_time_periods(USUiC_df, US_UiC_growing_season_2023)
USUiC_df_LE_filtered_2024 = filter_time_periods(USUiC_df, US_UiC_growing_season_2024)

# IN
USVT1_df_LE_filtered_2023 = filter_time_periods(USVT1_df, US_VT1_growing_season_2023)
USVT1_df_LE_filtered_2024 = filter_time_periods(USVT1_df, US_VT1_growing_season_2024)
USVT2_df_LE_filtered_2023 = filter_time_periods(USVT2_df, US_VT2_growing_season_2023)
USVT2_df_LE_filtered_2024 = filter_time_periods(USVT2_df, US_VT2_growing_season_2024)


# Replace all -9999 with np.nan values
# PA
USUC1_df_LE_filtered_2019.replace(-9999, np.nan, inplace=True)
USUC1_df_LE_filtered_2020.replace(-9999, np.nan, inplace=True)
USUC1_df_LE_filtered_2021.replace(-9999, np.nan, inplace=True)
USUC1_df_LE_filtered_2022.replace(-9999, np.nan, inplace=True)
USUC1_df_LE_filtered_2023.replace(-9999, np.nan, inplace=True)
USUC1_df_LE_filtered_2024.replace(-9999, np.nan, inplace=True)
USUC2_df_LE_filtered_2019.replace(-9999, np.nan, inplace=True)
USUC2_df_LE_filtered_2020.replace(-9999, np.nan, inplace=True)
USUC2_df_LE_filtered_2021.replace(-9999, np.nan, inplace=True)
USUC2_df_LE_filtered_2022.replace(-9999, np.nan, inplace=True)
USUC2_df_LE_filtered_2023.replace(-9999, np.nan, inplace=True)
USUC2_df_LE_filtered_2024.replace(-9999, np.nan, inplace=True)
USHWB_df_LE_filtered_2016.replace(-9999, np.nan, inplace=True)
USHWB_df_LE_filtered_2017.replace(-9999, np.nan, inplace=True)

# CA
USBi1_df_LE_filtered_2018.replace(-9999, np.nan, inplace=True)
USBi1_df_LE_filtered_2019.replace(-9999, np.nan, inplace=True)
USBi1_df_LE_filtered_2020.replace(-9999, np.nan, inplace=True)
USBi1_df_LE_filtered_2021.replace(-9999, np.nan, inplace=True)
USBi1_df_LE_filtered_2022.replace(-9999, np.nan, inplace=True)
USBi1_df_LE_filtered_2023.replace(-9999, np.nan, inplace=True)
USBi1_df_LE_filtered_2024.replace(-9999, np.nan, inplace=True)
USBi2_df_LE_filtered_2018.replace(-9999, np.nan, inplace=True)
USBi2_df_LE_filtered_2019.replace(-9999, np.nan, inplace=True)
USBi2_df_LE_filtered_2020.replace(-9999, np.nan, inplace=True)
USBi2_df_LE_filtered_2021.replace(-9999, np.nan, inplace=True)
USBi2_df_LE_filtered_2022.replace(-9999, np.nan, inplace=True)
USBi2_df_LE_filtered_2023.replace(-9999, np.nan, inplace=True)
USBi2_df_LE_filtered_2024.replace(-9999, np.nan, inplace=True)
USTw3_df_LE_filtered_2017.replace(-9999, np.nan, inplace=True)
USTw3_df_LE_filtered_2018.replace(-9999, np.nan, inplace=True)

# IL
USUiA_df_LE_filtered_2017.replace(-9999, np.nan, inplace=True)
USUiA_df_LE_filtered_2018.replace(-9999, np.nan, inplace=True)
USUiA_df_LE_filtered_2019.replace(-9999, np.nan, inplace=True)
USUiA_df_LE_filtered_2020.replace(-9999, np.nan, inplace=True)
USUiA_df_LE_filtered_2021.replace(-9999, np.nan, inplace=True)
USUiA_df_LE_filtered_2022.replace(-9999, np.nan, inplace=True)
USUiA_df_LE_filtered_2023.replace(-9999, np.nan, inplace=True)
USUiA_df_LE_filtered_2024.replace(-9999, np.nan, inplace=True)
USUiB_df_LE_filtered_2017.replace(-9999, np.nan, inplace=True)
USUiB_df_LE_filtered_2018.replace(-9999, np.nan, inplace=True)
USUiB_df_LE_filtered_2019.replace(-9999, np.nan, inplace=True)
USUiB_df_LE_filtered_2020.replace(-9999, np.nan, inplace=True)
USUiB_df_LE_filtered_2021.replace(-9999, np.nan, inplace=True)
USUiB_df_LE_filtered_2022.replace(-9999, np.nan, inplace=True)
USUiB_df_LE_filtered_2023.replace(-9999, np.nan, inplace=True)
USUiB_df_LE_filtered_2024.replace(-9999, np.nan, inplace=True)
USUiC_df_LE_filtered_2017.replace(-9999, np.nan, inplace=True)
USUiC_df_LE_filtered_2018.replace(-9999, np.nan, inplace=True)
USUiC_df_LE_filtered_2019.replace(-9999, np.nan, inplace=True)
USUiC_df_LE_filtered_2020.replace(-9999, np.nan, inplace=True)
USUiC_df_LE_filtered_2021.replace(-9999, np.nan, inplace=True)
USUiC_df_LE_filtered_2022.replace(-9999, np.nan, inplace=True)
USUiC_df_LE_filtered_2023.replace(-9999, np.nan, inplace=True)
USUiC_df_LE_filtered_2024.replace(-9999, np.nan, inplace=True)

# IN
USVT1_df_LE_filtered_2023.replace(-9999, np.nan, inplace=True)
USVT1_df_LE_filtered_2024.replace(-9999, np.nan, inplace=True)
USVT2_df_LE_filtered_2023.replace(-9999, np.nan, inplace=True)
USVT2_df_LE_filtered_2024.replace(-9999, np.nan, inplace=True)

C:\Users\adadkhah\AppData\Local\Temp\ipykernel_28248\3026315095.py:208: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  USUC1_df_LE_filtered_2019.replace(-9999, np.nan, inplace=True)
C:\Users\adadkhah\AppData\Local\Temp\ipykernel_28248\3026315095.py:209: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  USUC1_df_LE_filtered_2020.replace(-9999, np.nan, inplace=True)
C:\Users\adadkhah\AppData\Local\Temp\ipykernel_28248\3026315095.py:210: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexi

#### Half Hourly EB correction

In [46]:
# perform energy balance correction on half hourly values of LE and H
### PA ###
USUC1_df_2019_LE_H_Corrected = EBC_half_hourly_data(USUC1_df_LE_filtered_2019,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USUC1_df_2020_LE_H_Corrected = EBC_half_hourly_data(USUC1_df_LE_filtered_2020,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USUC1_df_2021_LE_H_Corrected = EBC_half_hourly_data(USUC1_df_LE_filtered_2021,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USUC1_df_2022_LE_H_Corrected = EBC_half_hourly_data(USUC1_df_LE_filtered_2022,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USUC1_df_2023_LE_H_Corrected = EBC_half_hourly_data(USUC1_df_LE_filtered_2023,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USUC1_df_2024_LE_H_Corrected = EBC_half_hourly_data(USUC1_df_LE_filtered_2024,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})

USUC2_df_2019_LE_H_Corrected = EBC_half_hourly_data(USUC2_df_LE_filtered_2019,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USUC2_df_2020_LE_H_Corrected = EBC_half_hourly_data(USUC2_df_LE_filtered_2020,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USUC2_df_2021_LE_H_Corrected = EBC_half_hourly_data(USUC2_df_LE_filtered_2021,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USUC2_df_2022_LE_H_Corrected = EBC_half_hourly_data(USUC2_df_LE_filtered_2022,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USUC2_df_2023_LE_H_Corrected = EBC_half_hourly_data(USUC2_df_LE_filtered_2023,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USUC2_df_2024_LE_H_Corrected = EBC_half_hourly_data(USUC2_df_LE_filtered_2024,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})

USHWB_df_2016_LE_H_Corrected = EBC_half_hourly_data(USHWB_df_LE_filtered_2016,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USHWB_df_2017_LE_H_Corrected = EBC_half_hourly_data(USHWB_df_LE_filtered_2017,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})

### CA ###
USBi1_df_2018_LE_H_Corrected = EBC_half_hourly_data(USBi1_df_LE_filtered_2018,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USBi1_df_2019_LE_H_Corrected = EBC_half_hourly_data(USBi1_df_LE_filtered_2019,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USBi1_df_2020_LE_H_Corrected = EBC_half_hourly_data(USBi1_df_LE_filtered_2020,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USBi1_df_2021_LE_H_Corrected = EBC_half_hourly_data(USBi1_df_LE_filtered_2021,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USBi1_df_2022_LE_H_Corrected = EBC_half_hourly_data(USBi1_df_LE_filtered_2022,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USBi1_df_2023_LE_H_Corrected = EBC_half_hourly_data(USBi1_df_LE_filtered_2023,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USBi1_df_2024_LE_H_Corrected = EBC_half_hourly_data(USBi1_df_LE_filtered_2024,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})

USBi2_df_2018_LE_H_Corrected = EBC_half_hourly_data(USBi2_df_LE_filtered_2018,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USBi2_df_2019_LE_H_Corrected = EBC_half_hourly_data(USBi2_df_LE_filtered_2019,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USBi2_df_2020_LE_H_Corrected = EBC_half_hourly_data(USBi2_df_LE_filtered_2020,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USBi2_df_2021_LE_H_Corrected = EBC_half_hourly_data(USBi2_df_LE_filtered_2021,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USBi2_df_2022_LE_H_Corrected = EBC_half_hourly_data(USBi2_df_LE_filtered_2022,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USBi2_df_2023_LE_H_Corrected = EBC_half_hourly_data(USBi2_df_LE_filtered_2023,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USBi2_df_2024_LE_H_Corrected = EBC_half_hourly_data(USBi2_df_LE_filtered_2024,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})

USTw3_df_2017_LE_H_Corrected = EBC_half_hourly_data(USTw3_df_LE_filtered_2017,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USTw3_df_2018_LE_H_Corrected = EBC_half_hourly_data(USTw3_df_LE_filtered_2018,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})

# IL
USUiA_df_2017_LE_H_Corrected = EBC_half_hourly_data(USUiA_df_LE_filtered_2017,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USUiA_df_2018_LE_H_Corrected = EBC_half_hourly_data(USUiA_df_LE_filtered_2018,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USUiA_df_2019_LE_H_Corrected = EBC_half_hourly_data(USUiA_df_LE_filtered_2019,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USUiA_df_2020_LE_H_Corrected = EBC_half_hourly_data(USUiA_df_LE_filtered_2020,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USUiA_df_2021_LE_H_Corrected = EBC_half_hourly_data(USUiA_df_LE_filtered_2021,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USUiA_df_2022_LE_H_Corrected = EBC_half_hourly_data(USUiA_df_LE_filtered_2022,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USUiA_df_2023_LE_H_Corrected = EBC_half_hourly_data(USUiA_df_LE_filtered_2023,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USUiA_df_2024_LE_H_Corrected = EBC_half_hourly_data(USUiA_df_LE_filtered_2024,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})

USUiA_df_2017_LE_H_Corrected = EBC_half_hourly_data(USUiB_df_LE_filtered_2017,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USUiB_df_2018_LE_H_Corrected = EBC_half_hourly_data(USUiB_df_LE_filtered_2018,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USUiB_df_2019_LE_H_Corrected = EBC_half_hourly_data(USUiB_df_LE_filtered_2019,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USUiB_df_2020_LE_H_Corrected = EBC_half_hourly_data(USUiB_df_LE_filtered_2020,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USUiB_df_2021_LE_H_Corrected = EBC_half_hourly_data(USUiB_df_LE_filtered_2021,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USUiB_df_2022_LE_H_Corrected = EBC_half_hourly_data(USUiB_df_LE_filtered_2022,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USUiB_df_2023_LE_H_Corrected = EBC_half_hourly_data(USUiB_df_LE_filtered_2023,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USUiB_df_2024_LE_H_Corrected = EBC_half_hourly_data(USUiB_df_LE_filtered_2024,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})

USUiC_df_2017_LE_H_Corrected = EBC_half_hourly_data(USUiC_df_LE_filtered_2017,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USUiC_df_2018_LE_H_Corrected = EBC_half_hourly_data(USUiC_df_LE_filtered_2018,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USUiC_df_2019_LE_H_Corrected = EBC_half_hourly_data(USUiC_df_LE_filtered_2019,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USUiC_df_2020_LE_H_Corrected = EBC_half_hourly_data(USUiC_df_LE_filtered_2020,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USUiC_df_2021_LE_H_Corrected = EBC_half_hourly_data(USUiC_df_LE_filtered_2021,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USUiC_df_2022_LE_H_Corrected = EBC_half_hourly_data(USUiC_df_LE_filtered_2022,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USUiC_df_2023_LE_H_Corrected = EBC_half_hourly_data(USUiC_df_LE_filtered_2023,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USUiC_df_2024_LE_H_Corrected = EBC_half_hourly_data(USUiC_df_LE_filtered_2024,
                                                       flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})

# IN
USVT1_df_2023_LE_H_Corrected = EBC_half_hourly_data(USVT1_df_LE_filtered_2023,
                                                    flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USVT1_df_2024_LE_H_Corrected = EBC_half_hourly_data(USVT1_df_LE_filtered_2024,
                                                    flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})

USVT2_df_2023_LE_H_Corrected = EBC_half_hourly_data(USVT2_df_LE_filtered_2023,
                                                    flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})
USVT2_df_2024_LE_H_Corrected = EBC_half_hourly_data(USVT2_df_LE_filtered_2024,
                                                    flux_cols_dict={'Rn':'NETRAD', 'G':'G','LE':'LE', 'H':'H'})


# only keep daylight hours based on Rn>0 (6 am to 6:30 pm based on local time)
### PA ###
USUC1_df_LE_filtered_2019_daylight = USUC1_df_2019_LE_H_Corrected.between_time("06:00", "18:30")
USUC1_df_LE_filtered_2020_daylight = USUC1_df_2020_LE_H_Corrected.between_time("06:00", "18:30")
USUC1_df_LE_filtered_2021_daylight = USUC1_df_2021_LE_H_Corrected.between_time("06:00", "18:30")
USUC1_df_LE_filtered_2022_daylight = USUC1_df_2022_LE_H_Corrected.between_time("06:00", "18:30")
USUC1_df_LE_filtered_2023_daylight = USUC1_df_2023_LE_H_Corrected.between_time("06:00", "18:30")
USUC1_df_LE_filtered_2024_daylight = USUC1_df_2024_LE_H_Corrected.between_time("06:00", "18:30")
USUC2_df_LE_filtered_2019_daylight = USUC2_df_2019_LE_H_Corrected.between_time("06:00", "18:30")
USUC2_df_LE_filtered_2020_daylight = USUC2_df_2020_LE_H_Corrected.between_time("06:00", "18:30")
USUC2_df_LE_filtered_2021_daylight = USUC2_df_2021_LE_H_Corrected.between_time("06:00", "18:30")
USUC2_df_LE_filtered_2022_daylight = USUC2_df_2022_LE_H_Corrected.between_time("06:00", "18:30")
USUC2_df_LE_filtered_2023_daylight = USUC2_df_2023_LE_H_Corrected.between_time("06:00", "18:30")
USUC2_df_LE_filtered_2024_daylight = USUC2_df_2024_LE_H_Corrected.between_time("06:00", "18:30")
USHWB_df_LE_filtered_2016_daylight = USHWB_df_2016_LE_H_Corrected.between_time("06:00", "18:30")
USHWB_df_LE_filtered_2017_daylight = USHWB_df_2017_LE_H_Corrected.between_time("06:00", "18:30")

### CA ###
USBi1_df_LE_filtered_2018_daylight = USBi1_df_2018_LE_H_Corrected.between_time("06:00", "18:30")
USBi1_df_LE_filtered_2019_daylight = USBi1_df_2019_LE_H_Corrected.between_time("06:00", "18:30")
USBi1_df_LE_filtered_2020_daylight = USBi1_df_2020_LE_H_Corrected.between_time("06:00", "18:30")
USBi1_df_LE_filtered_2021_daylight = USBi1_df_2021_LE_H_Corrected.between_time("06:00", "18:30")
USBi1_df_LE_filtered_2022_daylight = USBi1_df_2022_LE_H_Corrected.between_time("06:00", "18:30")
USBi1_df_LE_filtered_2023_daylight = USBi1_df_2023_LE_H_Corrected.between_time("06:00", "18:30")
USBi1_df_LE_filtered_2024_daylight = USBi1_df_2024_LE_H_Corrected.between_time("06:00", "18:30")
USBi2_df_LE_filtered_2018_daylight = USBi2_df_2018_LE_H_Corrected.between_time("06:00", "18:30")
USBi2_df_LE_filtered_2019_daylight = USBi2_df_2019_LE_H_Corrected.between_time("06:00", "18:30")
USBi2_df_LE_filtered_2020_daylight = USBi2_df_2020_LE_H_Corrected.between_time("06:00", "18:30")
USBi2_df_LE_filtered_2021_daylight = USBi2_df_2021_LE_H_Corrected.between_time("06:00", "18:30")
USBi2_df_LE_filtered_2022_daylight = USBi2_df_2022_LE_H_Corrected.between_time("06:00", "18:30")
USBi2_df_LE_filtered_2023_daylight = USBi2_df_2023_LE_H_Corrected.between_time("06:00", "18:30")
USBi2_df_LE_filtered_2024_daylight = USBi2_df_2024_LE_H_Corrected.between_time("06:00", "18:30")
USTw3_df_LE_filtered_2017_daylight = USTw3_df_2017_LE_H_Corrected.between_time("06:00", "18:30")
USTw3_df_LE_filtered_2018_daylight = USTw3_df_2018_LE_H_Corrected.between_time("06:00", "18:30")

### IL ###
USUiA_df_LE_filtered_2017_daylight = USUiA_df_2017_LE_H_Corrected.between_time("06:00", "18:30")                        
USUiA_df_LE_filtered_2018_daylight = USUiA_df_2018_LE_H_Corrected.between_time("06:00", "18:30")                          
USUiA_df_LE_filtered_2019_daylight = USUiA_df_2019_LE_H_Corrected.between_time("06:00", "18:30")                          
USUiA_df_LE_filtered_2020_daylight = USUiA_df_2020_LE_H_Corrected.between_time("06:00", "18:30")
USUiA_df_LE_filtered_2021_daylight = USUiA_df_2021_LE_H_Corrected.between_time("06:00", "18:30")                          
USUiA_df_LE_filtered_2022_daylight = USUiA_df_2022_LE_H_Corrected.between_time("06:00", "18:30")                          
USUiA_df_LE_filtered_2023_daylight = USUiA_df_2023_LE_H_Corrected.between_time("06:00", "18:30")                          
USUiA_df_LE_filtered_2024_daylight = USUiA_df_2024_LE_H_Corrected.between_time("06:00", "18:30")                          
USUiB_df_LE_filtered_2017_daylight = USUiA_df_2017_LE_H_Corrected.between_time("06:00", "18:30")                          
USUiB_df_LE_filtered_2018_daylight = USUiB_df_2018_LE_H_Corrected.between_time("06:00", "18:30")                          
USUiB_df_LE_filtered_2019_daylight = USUiB_df_2019_LE_H_Corrected.between_time("06:00", "18:30")                          
USUiB_df_LE_filtered_2020_daylight = USUiB_df_2020_LE_H_Corrected.between_time("06:00", "18:30")                         
USUiB_df_LE_filtered_2021_daylight = USUiB_df_2021_LE_H_Corrected.between_time("06:00", "18:30")                         
USUiB_df_LE_filtered_2022_daylight = USUiB_df_2022_LE_H_Corrected.between_time("06:00", "18:30")                           
USUiB_df_LE_filtered_2023_daylight = USUiB_df_2023_LE_H_Corrected.between_time("06:00", "18:30")                           
USUiB_df_LE_filtered_2024_daylight = USUiB_df_2024_LE_H_Corrected.between_time("06:00", "18:30")                           
USUiC_df_LE_filtered_2017_daylight = USUiC_df_2017_LE_H_Corrected.between_time("06:00", "18:30")                           
USUiC_df_LE_filtered_2018_daylight = USUiC_df_2018_LE_H_Corrected.between_time("06:00", "18:30")                           
USUiC_df_LE_filtered_2019_daylight = USUiC_df_2019_LE_H_Corrected.between_time("06:00", "18:30")                         
USUiC_df_LE_filtered_2020_daylight = USUiC_df_2020_LE_H_Corrected.between_time("06:00", "18:30")                          
USUiC_df_LE_filtered_2021_daylight = USUiC_df_2021_LE_H_Corrected.between_time("06:00", "18:30")                         
USUiC_df_LE_filtered_2022_daylight = USUiC_df_2022_LE_H_Corrected.between_time("06:00", "18:30")                         
USUiC_df_LE_filtered_2023_daylight = USUiC_df_2023_LE_H_Corrected.between_time("06:00", "18:30")                         
USUiC_df_LE_filtered_2024_daylight = USUiC_df_2024_LE_H_Corrected.between_time("06:00", "18:30")

### IN ###
USVT1_df_LE_filtered_2023_daylight = USVT1_df_2023_LE_H_Corrected.between_time("06:00", "18:30")
USVT1_df_LE_filtered_2024_daylight = USVT1_df_2024_LE_H_Corrected.between_time("06:00", "18:30")
USVT2_df_LE_filtered_2023_daylight = USVT2_df_2023_LE_H_Corrected.between_time("06:00", "18:30")
USVT2_df_LE_filtered_2024_daylight = USVT2_df_2024_LE_H_Corrected.between_time("06:00", "18:30")

C:\Users\adadkhah\AppData\Local\Temp\ipykernel_28248\3420895029.py:52: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  df["EBC_CF_Final"] = df["EBC_CF_Method1"].combine_first(df["EBC_CF_Method2"])
C:\Users\adadkhah\AppData\Local\Temp\ipykernel_28248\3420895029.py:52: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  df["EBC_CF_Final"] = df["EBC_CF_Method1"].combine_first(df["EBC_CF_Method2"])
C:\Users\adadkhah\AppData\Local\Temp\ipykernel_28248\3420895029.py:52: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future ve

In [47]:
USTw3_df_LE_filtered_2017_daylight

,NETRAD,LE,H,G,LE_SSITC_TEST,H_SSITC_TEST,TA,RH,SW_IN,LW_IN,...,hour,minute,time_of_day,EBC_CF,EBC_CF_Method1,EBC_CF_Method2,EBC_CF_Final,H_corr,LE_corr,Imb_corr
TIMESTAMP_START,,,,,,,,,,,,,,,,,,,,,
2017-01-21 06:00:00,NaN,13.651661,-25.134564,NaN,2.0,2.0,NaN,NaN,NaN,NaN,...,6,0,6.0,NaN,1.354974,NaN,1.354974,-34.056679,18.497645,NaN
2017-01-21 06:30:00,NaN,-2.553157,-7.242922,NaN,2.0,2.0,NaN,NaN,NaN,NaN,...,6,30,6.5,NaN,1.354974,NaN,1.354974,-9.813971,-3.459461,NaN
2017-01-21 07:00:00,NaN,1.704240,-11.058935,NaN,2.0,2.0,NaN,NaN,NaN,NaN,...,7,0,7.0,NaN,1.354974,NaN,1.354974,-14.984569,2.309201,NaN
2017-01-21 07:30:00,NaN,5.050500,-12.824855,NaN,2.0,2.0,NaN,NaN,NaN,NaN,...,7,30,7.5,NaN,1.354974,NaN,1.354974,-17.377344,6.843296,NaN
2017-01-21 08:00:00,NaN,12.537525,-9.139037,NaN,2.0,2.0,NaN,NaN,NaN,NaN,...,8,0,8.0,NaN,1.354974,NaN,1.354974,-12.383157,16.988020,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2017-10-10 16:30:00,-26.270771,33.796835,-46.280559,4.206840,2.0,1.0,23.77,22.81,80.195049,337.723043,...,16,30,16.5,2.441388,1.381499,NaN,1.381499,-63.936541,46.690290,-13.231360
2017-10-10 17:00:00,-53.637088,27.998044,-53.667416,-1.807550,1.0,1.0,22.93,22.45,49.362341,327.737985,...,17,0,17.0,2.019120,1.381499,NaN,1.381499,-74.141476,38.679267,-16.367329
2017-10-10 17:30:00,-85.832863,14.525189,-45.750655,-7.379879,2.0,2.0,21.40,25.53,0.600150,314.846002,...,17,30,17.5,2.512468,1.381499,NaN,1.381499,-63.204479,20.066533,-35.315037


In [48]:
# aggregate data to hourly values and only keep columns that are helpful for evapotranspiration calculation/prediction

### PA ###
agg_funcs = {
    'LE_corr':'mean',
    'TA':'mean',
    'RH':'mean',
    'SW_IN':'mean',
    'LW_IN':'mean',
    'WS':'mean',
    'PA':'mean',
    'P_RAIN':'sum'   
}

# only keep columns that are going to be used for the model training
Columns_to_keep = ['LE_corr', 'TA', 'RH', 'SW_IN',  'LW_IN','WS', 'PA', 'P_RAIN']

USUC1_df_processed_2019 = USUC1_df_LE_filtered_2019_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USUC1_df_processed_2020 = USUC1_df_LE_filtered_2020_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USUC1_df_processed_2021 = USUC1_df_LE_filtered_2021_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USUC1_df_processed_2022 = USUC1_df_LE_filtered_2022_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USUC1_df_processed_2023 = USUC1_df_LE_filtered_2023_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USUC1_df_processed_2024 = USUC1_df_LE_filtered_2024_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USUC2_df_processed_2019 = USUC2_df_LE_filtered_2019_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USUC2_df_processed_2020 = USUC2_df_LE_filtered_2020_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USUC2_df_processed_2021 = USUC2_df_LE_filtered_2021_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USUC2_df_processed_2022 = USUC2_df_LE_filtered_2022_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USUC2_df_processed_2023 = USUC2_df_LE_filtered_2023_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USUC2_df_processed_2024 = USUC2_df_LE_filtered_2024_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USHWB_df_processed_2016 = USHWB_df_LE_filtered_2016_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USHWB_df_processed_2017 = USHWB_df_LE_filtered_2017_daylight[Columns_to_keep].resample('h').agg(agg_funcs)

### CA ### 
USBi1_df_processed_2018 = USBi1_df_LE_filtered_2018_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USBi1_df_processed_2019 = USBi1_df_LE_filtered_2019_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USBi1_df_processed_2020 = USBi1_df_LE_filtered_2020_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USBi1_df_processed_2021 = USBi1_df_LE_filtered_2021_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USBi1_df_processed_2022 = USBi1_df_LE_filtered_2022_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USBi1_df_processed_2023 = USBi1_df_LE_filtered_2023_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USBi1_df_processed_2024 = USBi1_df_LE_filtered_2024_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USBi2_df_processed_2018 = USBi2_df_LE_filtered_2018_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USBi2_df_processed_2019 = USBi2_df_LE_filtered_2019_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USBi2_df_processed_2020 = USBi2_df_LE_filtered_2020_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USBi2_df_processed_2021 = USBi2_df_LE_filtered_2021_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USBi2_df_processed_2022 = USBi2_df_LE_filtered_2022_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USBi2_df_processed_2023 = USBi2_df_LE_filtered_2023_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USBi2_df_processed_2024 = USBi2_df_LE_filtered_2024_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USTw3_df_processed_2017 = USTw3_df_LE_filtered_2017_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USTw3_df_processed_2018 = USTw3_df_LE_filtered_2018_daylight[Columns_to_keep].resample('h').agg(agg_funcs)

### IL ###
USUiA_df_processed_2017 = USUiA_df_LE_filtered_2017_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USUiA_df_processed_2018 = USUiA_df_LE_filtered_2018_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USUiA_df_processed_2019 = USUiA_df_LE_filtered_2019_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USUiA_df_processed_2020 = USUiA_df_LE_filtered_2020_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USUiA_df_processed_2021 = USUiA_df_LE_filtered_2021_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USUiA_df_processed_2022 = USUiA_df_LE_filtered_2022_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USUiA_df_processed_2023 = USUiA_df_LE_filtered_2023_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USUiA_df_processed_2024 = USUiA_df_LE_filtered_2024_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USUiB_df_processed_2017 = USUiB_df_LE_filtered_2017_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USUiB_df_processed_2018 = USUiB_df_LE_filtered_2018_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USUiB_df_processed_2019 = USUiB_df_LE_filtered_2019_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USUiB_df_processed_2020 = USUiB_df_LE_filtered_2020_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USUiB_df_processed_2021 = USUiB_df_LE_filtered_2021_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USUiB_df_processed_2022 = USUiB_df_LE_filtered_2022_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USUiB_df_processed_2023 = USUiB_df_LE_filtered_2023_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USUiB_df_processed_2024 = USUiB_df_LE_filtered_2024_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USUiC_df_processed_2017 = USUiC_df_LE_filtered_2017_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USUiC_df_processed_2018 = USUiC_df_LE_filtered_2018_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USUiC_df_processed_2019 = USUiC_df_LE_filtered_2019_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USUiC_df_processed_2020 = USUiC_df_LE_filtered_2020_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USUiC_df_processed_2021 = USUiC_df_LE_filtered_2021_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USUiC_df_processed_2022 = USUiC_df_LE_filtered_2022_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USUiC_df_processed_2023 = USUiC_df_LE_filtered_2023_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USUiC_df_processed_2024 = USUiC_df_LE_filtered_2024_daylight[Columns_to_keep].resample('h').agg(agg_funcs)

### IN ###
USVT1_df_processed_2023 = USVT1_df_LE_filtered_2023_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USVT1_df_processed_2024 = USVT1_df_LE_filtered_2024_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USVT2_df_processed_2023 = USVT2_df_LE_filtered_2023_daylight[Columns_to_keep].resample('h').agg(agg_funcs)
USVT2_df_processed_2024 = USVT2_df_LE_filtered_2024_daylight[Columns_to_keep].resample('h').agg(agg_funcs)



# Now calculate ETa following the method of Harrison, L.P. 1963
TA_col = 'TA' # air temperature column
LE_col = 'LE_corr' # latent heat flux column

def calculate_hourly_ETa(processed_df, LE_column=LE_col, TA_column=TA_col):
    df = copy.deepcopy(processed_df)
    df['ETa_corr'] = (df[LE_col] * 60 * 60) / ((2.501 - 0.002361 * df[TA_col]) * 10**6)
    return df.drop([LE_col], axis=1)

### PA ###
USUC1_df_processed_2019_ETa = calculate_hourly_ETa(USUC1_df_processed_2019)
USUC1_df_processed_2020_ETa = calculate_hourly_ETa(USUC1_df_processed_2020)
USUC1_df_processed_2021_ETa = calculate_hourly_ETa(USUC1_df_processed_2021)
USUC1_df_processed_2022_ETa = calculate_hourly_ETa(USUC1_df_processed_2022)
USUC1_df_processed_2023_ETa = calculate_hourly_ETa(USUC1_df_processed_2023)
USUC1_df_processed_2024_ETa = calculate_hourly_ETa(USUC1_df_processed_2024)
USUC2_df_processed_2019_ETa = calculate_hourly_ETa(USUC2_df_processed_2019)
USUC2_df_processed_2020_ETa = calculate_hourly_ETa(USUC2_df_processed_2020)
USUC2_df_processed_2021_ETa = calculate_hourly_ETa(USUC2_df_processed_2021)
USUC2_df_processed_2022_ETa = calculate_hourly_ETa(USUC2_df_processed_2022)
USUC2_df_processed_2023_ETa = calculate_hourly_ETa(USUC2_df_processed_2023)
USUC2_df_processed_2024_ETa = calculate_hourly_ETa(USUC2_df_processed_2024)
USHWB_df_processed_2016_ETa = calculate_hourly_ETa(USHWB_df_processed_2016)
USHWB_df_processed_2017_ETa = calculate_hourly_ETa(USHWB_df_processed_2017)

### CA ###
USBi1_df_processed_2018_ETa = calculate_hourly_ETa(USBi1_df_processed_2018)
USBi1_df_processed_2019_ETa = calculate_hourly_ETa(USBi1_df_processed_2019)
USBi1_df_processed_2020_ETa = calculate_hourly_ETa(USBi1_df_processed_2020)
USBi1_df_processed_2021_ETa = calculate_hourly_ETa(USBi1_df_processed_2021)
USBi1_df_processed_2022_ETa = calculate_hourly_ETa(USBi1_df_processed_2022)
USBi1_df_processed_2023_ETa = calculate_hourly_ETa(USBi1_df_processed_2023)
USBi1_df_processed_2024_ETa = calculate_hourly_ETa(USBi1_df_processed_2024)
USBi2_df_processed_2018_ETa = calculate_hourly_ETa(USBi2_df_processed_2018)
USBi2_df_processed_2019_ETa = calculate_hourly_ETa(USBi2_df_processed_2019)
USBi2_df_processed_2020_ETa = calculate_hourly_ETa(USBi2_df_processed_2020)
USBi2_df_processed_2021_ETa = calculate_hourly_ETa(USBi2_df_processed_2021)
USBi2_df_processed_2022_ETa = calculate_hourly_ETa(USBi2_df_processed_2022)
USBi2_df_processed_2023_ETa = calculate_hourly_ETa(USBi2_df_processed_2023)
USBi2_df_processed_2024_ETa = calculate_hourly_ETa(USBi2_df_processed_2024)
USTw3_df_processed_2017_ETa = calculate_hourly_ETa(USTw3_df_processed_2017)
USTw3_df_processed_2018_ETa = calculate_hourly_ETa(USTw3_df_processed_2018)

### IL ###
USUiA_df_processed_2017_ETa = calculate_hourly_ETa(USUiA_df_processed_2017)
USUiA_df_processed_2018_ETa = calculate_hourly_ETa(USUiA_df_processed_2018)
USUiA_df_processed_2019_ETa = calculate_hourly_ETa(USUiA_df_processed_2019)
USUiA_df_processed_2020_ETa = calculate_hourly_ETa(USUiA_df_processed_2020)
USUiA_df_processed_2021_ETa = calculate_hourly_ETa(USUiA_df_processed_2021)
USUiA_df_processed_2022_ETa = calculate_hourly_ETa(USUiA_df_processed_2022)
USUiA_df_processed_2023_ETa = calculate_hourly_ETa(USUiA_df_processed_2023)
USUiA_df_processed_2024_ETa = calculate_hourly_ETa(USUiA_df_processed_2024)
USUiB_df_processed_2017_ETa = calculate_hourly_ETa(USUiB_df_processed_2017)
USUiB_df_processed_2018_ETa = calculate_hourly_ETa(USUiB_df_processed_2018)
USUiB_df_processed_2019_ETa = calculate_hourly_ETa(USUiB_df_processed_2019)
USUiB_df_processed_2020_ETa = calculate_hourly_ETa(USUiB_df_processed_2020)
USUiB_df_processed_2021_ETa = calculate_hourly_ETa(USUiB_df_processed_2021)
USUiB_df_processed_2022_ETa = calculate_hourly_ETa(USUiB_df_processed_2022)
USUiB_df_processed_2023_ETa = calculate_hourly_ETa(USUiB_df_processed_2023)
USUiB_df_processed_2024_ETa = calculate_hourly_ETa(USUiB_df_processed_2024)
USUiC_df_processed_2017_ETa = calculate_hourly_ETa(USUiC_df_processed_2017)
USUiC_df_processed_2018_ETa = calculate_hourly_ETa(USUiC_df_processed_2018)
USUiC_df_processed_2019_ETa = calculate_hourly_ETa(USUiC_df_processed_2019)
USUiC_df_processed_2020_ETa = calculate_hourly_ETa(USUiC_df_processed_2020)
USUiC_df_processed_2021_ETa = calculate_hourly_ETa(USUiC_df_processed_2021)
USUiC_df_processed_2022_ETa = calculate_hourly_ETa(USUiC_df_processed_2022)
USUiC_df_processed_2023_ETa = calculate_hourly_ETa(USUiC_df_processed_2023)
USUiC_df_processed_2024_ETa = calculate_hourly_ETa(USUiC_df_processed_2024)

### IN ###
USVT1_df_processed_2023_ETa = calculate_hourly_ETa(USVT1_df_processed_2023)
USVT1_df_processed_2024_ETa = calculate_hourly_ETa(USVT1_df_processed_2024)
USVT2_df_processed_2023_ETa = calculate_hourly_ETa(USVT2_df_processed_2023)
USVT2_df_processed_2024_ETa = calculate_hourly_ETa(USVT2_df_processed_2024)



# filter dates to only include growing seasons
def filter_time_periods(df, time_periods, buffer_days=0):
    buffer = pd.Timedelta(days=buffer_days)
    return df.loc[pd.Timestamp(time_periods[0][0]) - buffer : pd.Timestamp(time_periods[0][1]) + buffer]


### PA ###
USUC1_df_FilteredDates_2019_ETa = filter_time_periods(USUC1_df_processed_2019_ETa, US_UC_growing_season_2019).between_time("06:00", "18:00")
USUC1_df_FilteredDates_2020_ETa = filter_time_periods(USUC1_df_processed_2020_ETa, US_UC_growing_season_2020).between_time("06:00", "18:00")
USUC1_df_FilteredDates_2021_ETa = filter_time_periods(USUC1_df_processed_2021_ETa, US_UC_growing_season_2021).between_time("06:00", "18:00")
USUC1_df_FilteredDates_2022_ETa = filter_time_periods(USUC1_df_processed_2022_ETa, US_UC_growing_season_2022).between_time("06:00", "18:00")
USUC1_df_FilteredDates_2023_ETa = filter_time_periods(USUC1_df_processed_2023_ETa, US_UC_growing_season_2023).between_time("06:00", "18:00")
USUC1_df_FilteredDates_2024_ETa = filter_time_periods(USUC1_df_processed_2024_ETa, US_UC_growing_season_2024).between_time("06:00", "18:00")
USUC2_df_FilteredDates_2019_ETa = filter_time_periods(USUC2_df_processed_2019_ETa, US_UC_growing_season_2019).between_time("06:00", "18:00")
USUC2_df_FilteredDates_2020_ETa = filter_time_periods(USUC2_df_processed_2020_ETa, US_UC_growing_season_2020).between_time("06:00", "18:00")
USUC2_df_FilteredDates_2021_ETa = filter_time_periods(USUC2_df_processed_2021_ETa, US_UC_growing_season_2021).between_time("06:00", "18:00")
USUC2_df_FilteredDates_2022_ETa = filter_time_periods(USUC2_df_processed_2022_ETa, US_UC_growing_season_2022).between_time("06:00", "18:00")
USUC2_df_FilteredDates_2023_ETa = filter_time_periods(USUC2_df_processed_2023_ETa, US_UC_growing_season_2023).between_time("06:00", "18:00")
USUC2_df_FilteredDates_2024_ETa = filter_time_periods(USUC2_df_processed_2024_ETa, US_UC_growing_season_2024).between_time("06:00", "18:00")
USHWB_df_FilteredDates_2016_ETa = filter_time_periods(USHWB_df_processed_2016_ETa, US_HWB_growing_season_2016).between_time("06:00", "18:00")
USHWB_df_FilteredDates_2017_ETa = filter_time_periods(USHWB_df_processed_2017_ETa, US_HWB_growing_season_2017).between_time("06:00", "18:00")

### CA ###
USBi1_df_FilteredDates_2018_ETa = filter_time_periods(USBi1_df_processed_2018_ETa, US_Bi1_growing_season_2018).between_time("06:00", "18:00")
USBi1_df_FilteredDates_2019_ETa = filter_time_periods(USBi1_df_processed_2019_ETa, US_Bi1_growing_season_2019).between_time("06:00", "18:00")
USBi1_df_FilteredDates_2020_ETa = filter_time_periods(USBi1_df_processed_2020_ETa, US_Bi1_growing_season_2020).between_time("06:00", "18:00")
USBi1_df_FilteredDates_2021_ETa = filter_time_periods(USBi1_df_processed_2021_ETa, US_Bi1_growing_season_2021).between_time("06:00", "18:00")
USBi1_df_FilteredDates_2022_ETa = filter_time_periods(USBi1_df_processed_2022_ETa, US_Bi1_growing_season_2022).between_time("06:00", "18:00")
USBi1_df_FilteredDates_2023_ETa = filter_time_periods(USBi1_df_processed_2023_ETa, US_Bi1_growing_season_2023).between_time("06:00", "18:00")
USBi1_df_FilteredDates_2024_ETa = filter_time_periods(USBi1_df_processed_2024_ETa, US_Bi1_growing_season_2024).between_time("06:00", "18:00")
USBi2_df_FilteredDates_2018_ETa = filter_time_periods(USBi2_df_processed_2018_ETa, US_Bi2_growing_season_2018).between_time("06:00", "18:00")
USBi2_df_FilteredDates_2019_ETa = filter_time_periods(USBi2_df_processed_2019_ETa, US_Bi2_growing_season_2019).between_time("06:00", "18:00")
USBi2_df_FilteredDates_2020_ETa = filter_time_periods(USBi2_df_processed_2020_ETa, US_Bi2_growing_season_2020).between_time("06:00", "18:00")
USBi2_df_FilteredDates_2021_ETa = filter_time_periods(USBi2_df_processed_2021_ETa, US_Bi2_growing_season_2021).between_time("06:00", "18:00")
USBi2_df_FilteredDates_2022_ETa = filter_time_periods(USBi2_df_processed_2022_ETa, US_Bi2_growing_season_2022).between_time("06:00", "18:00")
USBi2_df_FilteredDates_2023_ETa = filter_time_periods(USBi2_df_processed_2023_ETa, US_Bi2_growing_season_2023).between_time("06:00", "18:00")
USBi2_df_FilteredDates_2024_ETa = filter_time_periods(USBi2_df_processed_2024_ETa, US_Bi2_growing_season_2024).between_time("06:00", "18:00")
USTw3_df_FilteredDates_2017_ETa = filter_time_periods(USTw3_df_processed_2017_ETa, US_Tw3_growing_season_2017).between_time("06:00", "18:00")
USTw3_df_FilteredDates_2018_ETa = filter_time_periods(USTw3_df_processed_2018_ETa, US_Tw3_growing_season_2018).between_time("06:00", "18:00")

### IL ###
USUiA_df_FilteredDates_2017_ETa = filter_time_periods(USUiA_df_processed_2017_ETa, US_UiA_growing_season_2017).between_time("06:00", "18:00")
USUiA_df_FilteredDates_2018_ETa = filter_time_periods(USUiA_df_processed_2018_ETa, US_UiA_growing_season_2018).between_time("06:00", "18:00")
USUiA_df_FilteredDates_2019_ETa = filter_time_periods(USUiA_df_processed_2019_ETa, US_UiA_growing_season_2019).between_time("06:00", "18:00")
USUiA_df_FilteredDates_2020_ETa = filter_time_periods(USUiA_df_processed_2020_ETa, US_UiA_growing_season_2020).between_time("06:00", "18:00")
USUiA_df_FilteredDates_2021_ETa = filter_time_periods(USUiA_df_processed_2021_ETa, US_UiA_growing_season_2021).between_time("06:00", "18:00")
USUiA_df_FilteredDates_2022_ETa = filter_time_periods(USUiA_df_processed_2022_ETa, US_UiA_growing_season_2022).between_time("06:00", "18:00")
USUiA_df_FilteredDates_2023_ETa = filter_time_periods(USUiA_df_processed_2023_ETa, US_UiA_growing_season_2023).between_time("06:00", "18:00")
USUiA_df_FilteredDates_2024_ETa = filter_time_periods(USUiA_df_processed_2024_ETa, US_UiA_growing_season_2024).between_time("06:00", "18:00")
USUiB_df_FilteredDates_2017_ETa = filter_time_periods(USUiB_df_processed_2017_ETa, US_UiB_growing_season_2017).between_time("06:00", "18:00")
USUiB_df_FilteredDates_2018_ETa = filter_time_periods(USUiB_df_processed_2018_ETa, US_UiB_growing_season_2018).between_time("06:00", "18:00")
USUiB_df_FilteredDates_2019_ETa = filter_time_periods(USUiB_df_processed_2019_ETa, US_UiB_growing_season_2019).between_time("06:00", "18:00")
USUiB_df_FilteredDates_2020_ETa = filter_time_periods(USUiB_df_processed_2020_ETa, US_UiB_growing_season_2020).between_time("06:00", "18:00")
USUiB_df_FilteredDates_2021_ETa = filter_time_periods(USUiB_df_processed_2021_ETa, US_UiB_growing_season_2021).between_time("06:00", "18:00")
USUiB_df_FilteredDates_2022_ETa = filter_time_periods(USUiB_df_processed_2022_ETa, US_UiB_growing_season_2022).between_time("06:00", "18:00")
USUiB_df_FilteredDates_2023_ETa = filter_time_periods(USUiB_df_processed_2023_ETa, US_UiB_growing_season_2023).between_time("06:00", "18:00")
USUiB_df_FilteredDates_2024_ETa = filter_time_periods(USUiB_df_processed_2024_ETa, US_UiB_growing_season_2024).between_time("06:00", "18:00")
USUiC_df_FilteredDates_2017_ETa = filter_time_periods(USUiC_df_processed_2017_ETa, US_UiC_growing_season_2017).between_time("06:00", "18:00")
USUiC_df_FilteredDates_2018_ETa = filter_time_periods(USUiC_df_processed_2018_ETa, US_UiC_growing_season_2018).between_time("06:00", "18:00")
USUiC_df_FilteredDates_2019_ETa = filter_time_periods(USUiC_df_processed_2019_ETa, US_UiC_growing_season_2019).between_time("06:00", "18:00")
USUiC_df_FilteredDates_2020_ETa = filter_time_periods(USUiC_df_processed_2020_ETa, US_UiC_growing_season_2020).between_time("06:00", "18:00")
USUiC_df_FilteredDates_2021_ETa = filter_time_periods(USUiC_df_processed_2021_ETa, US_UiC_growing_season_2021).between_time("06:00", "18:00")
USUiC_df_FilteredDates_2022_ETa = filter_time_periods(USUiC_df_processed_2022_ETa, US_UiC_growing_season_2022).between_time("06:00", "18:00")
USUiC_df_FilteredDates_2023_ETa = filter_time_periods(USUiC_df_processed_2023_ETa, US_UiC_growing_season_2023).between_time("06:00", "18:00")
USUiC_df_FilteredDates_2024_ETa = filter_time_periods(USUiC_df_processed_2024_ETa, US_UiC_growing_season_2024).between_time("06:00", "18:00")

### IN ###
USVT1_df_FilteredDates_2023_ETa = filter_time_periods(USVT1_df_processed_2023_ETa, US_VT1_growing_season_2023).between_time("06:00", "18:00")
USVT1_df_FilteredDates_2024_ETa = filter_time_periods(USVT1_df_processed_2024_ETa, US_VT1_growing_season_2024).between_time("06:00", "18:00")
USVT2_df_FilteredDates_2023_ETa = filter_time_periods(USVT2_df_processed_2023_ETa, US_VT2_growing_season_2023).between_time("06:00", "18:00")
USVT2_df_FilteredDates_2024_ETa = filter_time_periods(USVT2_df_processed_2024_ETa, US_VT2_growing_season_2024).between_time("06:00", "18:00")

In [49]:
# export the data
### PA ###
USUC1_df_FilteredDates_2019_ETa.to_csv('USUC1_df_2019_ETa_MET.csv')
USUC1_df_FilteredDates_2020_ETa.to_csv('USUC1_df_2020_ETa_MET.csv')
USUC1_df_FilteredDates_2021_ETa.to_csv('USUC1_df_2021_ETa_MET.csv')
USUC1_df_FilteredDates_2022_ETa.to_csv('USUC1_df_2022_ETa_MET.csv')
USUC1_df_FilteredDates_2023_ETa.to_csv('USUC1_df_2023_ETa_MET.csv')
USUC1_df_FilteredDates_2024_ETa.to_csv('USUC1_df_2024_ETa_MET.csv')
USUC2_df_FilteredDates_2019_ETa.to_csv('USUC2_df_2019_ETa_MET.csv')
USUC2_df_FilteredDates_2020_ETa.to_csv('USUC2_df_2020_ETa_MET.csv')
USUC2_df_FilteredDates_2021_ETa.to_csv('USUC2_df_2021_ETa_MET.csv')
USUC2_df_FilteredDates_2022_ETa.to_csv('USUC2_df_2022_ETa_MET.csv')
USUC2_df_FilteredDates_2023_ETa.to_csv('USUC2_df_2023_ETa_MET.csv')
USUC2_df_FilteredDates_2024_ETa.to_csv('USUC2_df_2024_ETa_MET.csv')
USHWB_df_FilteredDates_2016_ETa.to_csv('USHWB_df_2016_ETa_MET.csv')
USHWB_df_FilteredDates_2017_ETa.to_csv('USHWB_df_2017_ETa_MET.csv')

### CA ###
USBi1_df_FilteredDates_2018_ETa.to_csv('USBi1_df_2018_ETa_MET.csv')
USBi1_df_FilteredDates_2019_ETa.to_csv('USBi1_df_2019_ETa_MET.csv')
USBi1_df_FilteredDates_2020_ETa.to_csv('USBi1_df_2020_ETa_MET.csv')
USBi1_df_FilteredDates_2021_ETa.to_csv('USBi1_df_2021_ETa_MET.csv')
USBi1_df_FilteredDates_2022_ETa.to_csv('USBi1_df_2022_ETa_MET.csv')
USBi1_df_FilteredDates_2023_ETa.to_csv('USBi1_df_2023_ETa_MET.csv')
USBi1_df_FilteredDates_2024_ETa.to_csv('USBi1_df_2024_ETa_MET.csv')
USBi2_df_FilteredDates_2018_ETa.to_csv('USBi2_df_2018_ETa_MET.csv')
USBi2_df_FilteredDates_2019_ETa.to_csv('USBi2_df_2019_ETa_MET.csv')
USBi2_df_FilteredDates_2020_ETa.to_csv('USBi2_df_2020_ETa_MET.csv')
USBi2_df_FilteredDates_2021_ETa.to_csv('USBi2_df_2021_ETa_MET.csv')
USBi2_df_FilteredDates_2022_ETa.to_csv('USBi2_df_2022_ETa_MET.csv')
USBi2_df_FilteredDates_2023_ETa.to_csv('USBi2_df_2023_ETa_MET.csv')
USBi2_df_FilteredDates_2024_ETa.to_csv('USBi2_df_2024_ETa_MET.csv')
USTw3_df_FilteredDates_2017_ETa.to_csv('USTw3_df_2017_ETa_MET.csv')
USTw3_df_FilteredDates_2018_ETa.to_csv('USTw3_df_2018_ETa_MET.csv')

### IL ###
USUiA_df_FilteredDates_2017_ETa.to_csv('USUiA_df_2017_ETa_MET.csv')
USUiA_df_FilteredDates_2018_ETa.to_csv('USUiA_df_2018_ETa_MET.csv')
USUiA_df_FilteredDates_2019_ETa.to_csv('USUiA_df_2019_ETa_MET.csv')
USUiA_df_FilteredDates_2020_ETa.to_csv('USUiA_df_2020_ETa_MET.csv')
USUiA_df_FilteredDates_2021_ETa.to_csv('USUiA_df_2021_ETa_MET.csv')
USUiA_df_FilteredDates_2022_ETa.to_csv('USUiA_df_2022_ETa_MET.csv')
USUiA_df_FilteredDates_2023_ETa.to_csv('USUiA_df_2023_ETa_MET.csv')
USUiA_df_FilteredDates_2024_ETa.to_csv('USUiA_df_2024_ETa_MET.csv')
USUiB_df_FilteredDates_2017_ETa.to_csv('USUiB_df_2017_ETa_MET.csv')
USUiB_df_FilteredDates_2018_ETa.to_csv('USUiB_df_2018_ETa_MET.csv')
USUiB_df_FilteredDates_2019_ETa.to_csv('USUiB_df_2019_ETa_MET.csv')
USUiB_df_FilteredDates_2020_ETa.to_csv('USUiB_df_2020_ETa_MET.csv')
USUiB_df_FilteredDates_2021_ETa.to_csv('USUiB_df_2021_ETa_MET.csv')
USUiB_df_FilteredDates_2022_ETa.to_csv('USUiB_df_2022_ETa_MET.csv')
USUiB_df_FilteredDates_2023_ETa.to_csv('USUiB_df_2023_ETa_MET.csv')
USUiB_df_FilteredDates_2024_ETa.to_csv('USUiB_df_2024_ETa_MET.csv')
USUiC_df_FilteredDates_2017_ETa.to_csv('USUiC_df_2017_ETa_MET.csv')
USUiC_df_FilteredDates_2018_ETa.to_csv('USUiC_df_2018_ETa_MET.csv')
USUiC_df_FilteredDates_2019_ETa.to_csv('USUiC_df_2019_ETa_MET.csv')
USUiC_df_FilteredDates_2020_ETa.to_csv('USUiC_df_2020_ETa_MET.csv')
USUiC_df_FilteredDates_2021_ETa.to_csv('USUiC_df_2021_ETa_MET.csv')
USUiC_df_FilteredDates_2022_ETa.to_csv('USUiC_df_2022_ETa_MET.csv')
USUiC_df_FilteredDates_2023_ETa.to_csv('USUiC_df_2023_ETa_MET.csv')
USUiC_df_FilteredDates_2024_ETa.to_csv('USUiC_df_2024_ETa_MET.csv')

### IN ###
USVT1_df_FilteredDates_2023_ETa.to_csv('USVT1_df_2023_ETa_MET.csv')
USVT1_df_FilteredDates_2024_ETa.to_csv('USVT1_df_2024_ETa_MET.csv')
USVT2_df_FilteredDates_2023_ETa.to_csv('USVT2_df_2023_ETa_MET.csv')
USVT2_df_FilteredDates_2024_ETa.to_csv('USVT2_df_2024_ETa_MET.csv')